# Dunnhumby seed43: 유효성 처리 원형 M4·선형 N/V M5
기존 M1·보완 M4·선형 N/V M5는 검증 후 재사용합니다. 새 학습은 유효성 처리 원형 M4와 이를 결합한 선형 N/V M5 두 개입니다. M2의 affine/ID L2=1e-3, rho=.05, 64차원·2층·K=1·binary graph를 유지합니다. N/V 입력은 표현 안에서 공동학습하며 M4의 q_C는 손실 가중에 들어갑니다. 동결/외부 재정렬 없음.
최대300 epoch, 개발평가25마다, 전체 가격·구매금액 가중 적중값@10 최고값 선택(동률은 이른 시점), 100 이후4회 미개선 종료. 선택은100 이전일 수 있고 가장 이른 종료는200입니다. 원형 M4의 과거 고정100 결과는 이 선택규칙과 달라 재사용하지 않습니다. 동일 실험의 완료 캐시와 epoch 체크포인트는 재사용/재개합니다.
전체 경제지표@10에서 M5−원형 M4 및 M5−M1을 각각 보고합니다. M1 대비 여섯 Recall/NDCG 각각99% 보호, @20/@50·전 CLV구간·노출 전체 저장. 단일 개발시드: 유의성·일반화·CLV귀속 주장 없음. 최종 test/holdout 없음.
수정 이유: 기존 원형 M4는 무효 상품의 item_bin=-1을 마지막 가격구간으로 읽고 기본 가격백분위0.5를 사용해 추가 가중치를 줄 수 있었습니다. 수정 원형은 CLV·사용자 경제정보·상품 경제정보 중 하나라도 무효인 행의 정규화 전 가중치를1로 두고, 유효행에서는 원래 식을 그대로 적용합니다. 전체 학습행 평균으로 정규화합니다. 두 식의 차이를 진단에 기록합니다. 이전 원형 M4와 다른 모델 ID·결과 경로를 사용합니다. 처음에는 2단계 진단을 확인한 뒤 학습을 실행하세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,subprocess,json
SOURCE_COMMIT='df4ed5122c4fae358011ac85bbb4cf4163c4c7ab'
REPO=Path('/content/clv-original-m4-'+SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent==REPO.resolve(), '별도 런타임 또는 세션 재시작 필요'
os.chdir(REPO);sys.path.insert(0,str(REPO))
import clv_linear_nv_original_m4_screen as screen
import pandas as pd
ROOT=Path('/content/drive/MyDrive/논문/data')
REPORT=ROOT/'results_v3_dunnhumby_history_linear_nv_es_v2/reports/30287cad8e5ed1d1/result.json'
OUT=ROOT/'results_v3_dunnhumby_linear_nv_original_m4_validity_masked_seed43_v2'


## 2. 무효 입력 진단 — 학습 없음
기존 원형, 유효성 처리 원형, 기존 보완 가중식을 함께 계산합니다. 그룹은 서로 겹칩니다. normalized_weight_share는 학습행 가중총량 비율이지 gradient/성과 기여가 아닙니다. 두 원형 식의 무효행 추가 가중치 차이를 확인하세요.


In [ ]:
cfg,prepared,audit=screen.prepare(REPORT,OUT)
print(audit.to_string(index=False))
print(json.dumps(prepared['m4_diagnostics'],ensure_ascii=False,indent=2))
from zipfile import ZipFile,ZIP_DEFLATED
from google.colab import files
with ZipFile('/content/original_m4_validity_masked_audit_v2.zip','w',compression=ZIP_DEFLATED) as z:
    for name in ('m4_validity_audit.csv','m4_validity_audit.json'):z.write(OUT/name,arcname=name)
files.download('/content/original_m4_validity_masked_audit_v2.zip')


## 3. 유효성 처리 원형 M4·M5 학습
최대600 model-epoch의 새 학습입니다. 300 epoch를 두 모델이 나누는 것이 아닙니다. 두 모델 각각 하나의 optimizer로 처음부터 학습합니다. epoch마다 optimizer/난수상태를 저장하고 중단 후 재실행하면 이어갑니다. 호환 기준 결과 누락 시 자동 재학습하지 않습니다. 실시간 비용은 첫 모델의 epoch 로그로 추정하세요.


In [ ]:
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
paths=screen.run(cfg,prepared)
print(json.dumps(paths,ensure_ascii=False,indent=2))


In [ ]:
absolute=pd.read_csv(paths['absolute']);comparison=pd.read_csv(paths['comparison'])
metrics=['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50','price_purchase_amount_weighted_hit@10','vndcg@10','price_purchase_amount_weighted_hit@20','vndcg@20','price_purchase_amount_weighted_hit@50','vndcg@50']
print(absolute[['model_id','selected_epoch','stopped_epoch']+metrics].set_index('model_id').T.to_string())
print(comparison[comparison.metric.isin(metrics)].to_string(index=False))
with ZipFile('/content/linear_nv_original_m4_validity_masked_seed43_v2_results.zip','w',compression=ZIP_DEFLATED) as z:
    for path in paths.values():z.write(path,arcname=Path(path).name)
    for name in ('m4_validity_audit.csv','m4_validity_audit.json'):z.write(OUT/name,arcname=name)
files.download('/content/linear_nv_original_m4_validity_masked_seed43_v2_results.zip')
